# EDA — Predicting Student Health Risk (Playground Series S6E7)

Exploratory analysis of the training data to inform preprocessing and
feature-encoding decisions used in `src/preprocess.py`.

**Target**: `health_condition` — multi-class (`at-risk`, `fit`, `unhealthy`)
**Metric**: Balanced Accuracy
**Dataset**: 690,088 train rows / 295,753 test rows, 13 features (7 numeric, 6 categorical)

In [ ]:
# Load data into pandas
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e7/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e7/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

In [ ]:
# Overview of train set
#train_df.head()
#train_df.describe()
#train_df.describe(include='object')
#train_df.info()
#train_df.shape
#train_df.isnull().sum()

# Overview of test set
#test_df.info()
#test_df.isnull().sum()
#test_df.shape

In [ ]:
RANDOM_STATE = 42
TARGET = "health_condition"
ID_COL = "id"

numeric_cols = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
categorical_cols = [
    "diet_type", "stress_level", "sleep_quality",
    "physical_activity_level", "smoking_alcohol", "gender",
]

In [ ]:
## Exploratory analysis

The functions below cover, in order:
1. **Target class balance** — severe imbalance is expected, motivating
   balanced accuracy-aware training later (class weighting, balanced-accuracy
   early stopping in `src/train.py`).
2. **Missing value patterns** — checking whether missingness correlates
   with the target (informative missingness) or appears random (MCAR-like),
   which determines whether missing values should be imputed, flagged, or
   left native for the tree model to route.
3. **Numeric feature distributions** — skew, IQR outlier rate.
4. **Categorical features vs target** — class-rate breakdown per category.
5. **Sanity checks** — duplicate rows, duplicate IDs, train/test ID overlap.

In [ ]:
numeric_cols = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
categorical_cols = [
    "diet_type", "stress_level", "sleep_quality",
    "physical_activity_level", "smoking_alcohol", "gender",
]


def report_target_balance(df: pd.DataFrame, target_col: str) -> pd.Series:
    counts = df[target_col].value_counts()
    ratios = df[target_col].value_counts(normalize=True)
    print("=== Target class balance ===")
    print(pd.DataFrame({"count": counts, "ratio": ratios.round(4)}))
    print()
    return ratios


def report_missing_vs_target(df: pd.DataFrame, cols: list[str], target_col: str) -> pd.DataFrame:
    overall_dist = df[target_col].value_counts(normalize=True)
    class_labels = overall_dist.index.tolist()
    rows = []
    for col in cols:
        is_missing = df[col].isna()
        missing_count = int(is_missing.sum())
        row = {"column": col, "missing_count": missing_count, "missing_pct": is_missing.mean() * 100}
        dist_missing = df.loc[is_missing, target_col].value_counts(normalize=True) if missing_count > 0 else pd.Series(dtype=float)
        for cls in class_labels:
            row[f"rate_missing_{cls}"] = dist_missing.get(cls, np.nan)
            row[f"rate_overall_{cls}"] = overall_dist[cls]
        rows.append(row)
    result_df = pd.DataFrame(rows).sort_values("missing_pct", ascending=False)
    print("=== Missing value pattern vs target (multi-class) ===")
    print(f"Overall target distribution:\n{overall_dist.round(4)}\n")
    print(result_df.round(4).to_string(index=False))
    print()
    return result_df


def report_missing_cooccurrence(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    missing_matrix = df[cols].isna().astype(int)
    corr = missing_matrix.corr()
    print("=== Missing co-occurrence correlation ===")
    print(corr.round(2))
    print()
    return corr


def report_numeric_distribution(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        series = df[col].dropna()
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = ((series < lower_bound) | (series > upper_bound)).sum()
        rows.append({
            "column": col, "mean": series.mean(), "median": series.median(), "std": series.std(),
            "skew": series.skew(), "min": series.min(), "max": series.max(),
            "iqr_outlier_count": outlier_count, "iqr_outlier_pct": outlier_count / len(series) * 100,
        })
    result_df = pd.DataFrame(rows)
    print("=== Numeric distribution summary ===")
    print(result_df.round(3).to_string(index=False))
    print()
    return result_df


def report_numeric_vs_target(df: pd.DataFrame, cols: list[str], target_col: str) -> None:
    print("=== Numeric feature mean by target class ===")
    print(df.groupby(target_col)[cols].mean().round(3))
    print()
    print("=== Feature-feature correlation matrix (multicollinearity check) ===")
    print(df[cols].corr().round(2))
    print()


def report_categorical_vs_target(df: pd.DataFrame, cols: list[str], target_col: str) -> None:
    for col in cols:
        print(f"=== {col} vs target (row %) ===")
        crosstab = pd.crosstab(df[col], df[target_col], normalize="index")
        counts = df[col].value_counts(dropna=False)
        print(crosstab.round(3))
        print(f"-- counts per category (incl. NaN): \n{counts}")
        print()


def report_sanity_checks(train_df: pd.DataFrame, test_df: pd.DataFrame, id_col: str) -> None:
    print("=== Sanity checks ===")
    print(f"Duplicate rows in train (excluding id): {train_df.drop(columns=[id_col]).duplicated().sum()}")
    print(f"Duplicate id in train: {train_df[id_col].duplicated().sum()}")
    print(f"Overlap id between train and test: {len(set(train_df[id_col]) & set(test_df[id_col]))}")
    print()


# -----------------------------
# Run all EDA steps
# -----------------------------
target_ratios = report_target_balance(train_df, TARGET)
missing_vs_target_df = report_missing_vs_target(train_df, numeric_cols + categorical_cols, TARGET)
missing_corr_df = report_missing_cooccurrence(train_df, numeric_cols + categorical_cols)
numeric_dist_df = report_numeric_distribution(train_df, numeric_cols)
report_numeric_vs_target(train_df, numeric_cols, TARGET)
report_categorical_vs_target(train_df, categorical_cols, TARGET)
report_sanity_checks(train_df, test_df, ID_COL)

## Verifying ordinal assumptions

`stress_level`, `sleep_quality`, and `physical_activity_level` appear to
have a natural order. Before encoding them as ordinal integers in
`src/preprocess.py`, we verify the hypothesized order actually produces a
monotonic trend in target class rates (Spearman correlation between rank
and per-class rate). Only columns that pass this check use ordinal
encoding — the rest fall back to plain label encoding.

In [ ]:
def verify_ordinal_assumption(df: pd.DataFrame, col: str, order: list[str], target_col: str) -> bool:
    """
    Verify whether a hypothesized ordinal order for `col` produces monotonic
    trends in target class rates (ignoring NaN rows). With len(order) points
    per class, a perfectly monotonic trend gives |spearman rho| == 1.0.
    """
    subset = df[df[col].notna()]
    crosstab = pd.crosstab(subset[col], subset[target_col], normalize="index").reindex(order)

    print(f"=== Verifying order for '{col}': {order} ===")
    print(crosstab.round(4))

    class_results = {}
    for cls in crosstab.columns:
        rho, _ = spearmanr(range(len(order)), crosstab[cls].values)
        class_results[cls] = rho
        status = "MONOTONIC" if abs(round(rho, 6)) == 1.0 else "NOT monotonic"
        print(f"  class '{cls}': spearman rho = {rho:.3f}  -> {status}")

    is_fully_monotonic = all(abs(round(r, 6)) == 1.0 for r in class_results.values())
    print(f"  => {'VERIFIED' if is_fully_monotonic else 'REJECTED'}\n")
    return is_fully_monotonic


hypothesized_orders = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level": ["sedentary", "moderate", "active"],
}

ordinal_verified = {
    col: verify_ordinal_assumption(train_df, col, order, TARGET)
    for col, order in hypothesized_orders.items()
}

print("Verification summary:", ordinal_verified)

## Summary of findings → preprocessing decisions

- **Class imbalance**: at-risk 85.9% / unhealthy 8.4% / fit 5.8% —
  addressed via `class_weight="balanced"` and balanced-accuracy-based
  early stopping in training.
- **Missing values**: mostly MCAR-like (missing rate uncorrelated with
  target); no imputation performed — left native for XGBoost to route.
  `bmi` shows a mild deviation but not strong enough to warrant special
  handling.
- **Strongest predictors**: `stress_level` and `physical_activity_level`
  show the strongest (though non-monotonic) relationship with the target.
- **Ordinal verification**: only `sleep_quality` passed the monotonicity
  check (poor → average → good is fully monotonic across all 3 classes).
  `stress_level` and `physical_activity_level` are encoded as plain labels
  instead of ordinal, since their relationship with the target is not
  monotonic (e.g. `low` stress shows unexpectedly high `fit` rate).
- **Sanity checks**: no duplicate rows, no duplicate IDs, no train/test
  ID overlap.

See `src/preprocess.py` for the resulting encoding logic.